ML Breach Prediction Model
 Goal: Predict IF a ticket is a breach

In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

STEP 1: PREPARE DATA FOR ML

In [6]:
df = pd.read_csv('it_support_data.csv')

In [7]:
df.head(5)

,ticket_id,date,category,priority,resolution_time,repeated_issue,after_hours,is_breach
0,T0001,2023-02-22,Data Export,Low,4,False,True,True
1,T0002,2023-11-05,Data Export,Low,55,True,True,True
2,T0003,2023-01-04,Login Issue,Medium,70,False,False,False
3,T0004,2023-02-22,Malware Alert,Critical,44,True,True,False
4,T0005,2023-01-23,Login Issue,Critical,13,False,False,False


In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 8 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   ticket_id        1000 non-null   object
 1   date             1000 non-null   object
 2   category         1000 non-null   object
 3   priority         1000 non-null   object
 4   resolution_time  1000 non-null   int64 
 5   repeated_issue   1000 non-null   bool  
 6   after_hours      1000 non-null   bool  
 7   is_breach        1000 non-null   bool  
dtypes: bool(3), int64(1), object(4)
memory usage: 42.1+ KB


here the is_breach,after_hours,repeated_issue are in boolean converted them to int

In [9]:
# Convert Yes/No columns to 1/0 (ML needs numbers, not text)
df['is_breach'] = df['is_breach'].astype(int)
df['after_hours'] = df['after_hours'].astype(int)
df['repeated_issue'] = df['repeated_issue'].astype(int)

In [10]:
df.head(10)

,ticket_id,date,category,priority,resolution_time,repeated_issue,after_hours,is_breach
0,T0001,2023-02-22,Data Export,Low,4,0,1,1
1,T0002,2023-11-05,Data Export,Low,55,1,1,1
2,T0003,2023-01-04,Login Issue,Medium,70,0,0,0
3,T0004,2023-02-22,Malware Alert,Critical,44,1,1,0
4,T0005,2023-01-23,Login Issue,Critical,13,0,0,0
5,T0006,2023-05-31,Data Export,Critical,69,1,0,1
6,T0007,2023-05-29,Data Export,High,25,0,1,1
7,T0008,2023-03-25,Login Issue,Medium,13,0,0,0
8,T0009,2023-02-06,Access Request,High,27,0,0,0
9,T0010,2023-11-24,Phishing Email,Medium,69,0,1,1


In [11]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 8 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   ticket_id        1000 non-null   object
 1   date             1000 non-null   object
 2   category         1000 non-null   object
 3   priority         1000 non-null   object
 4   resolution_time  1000 non-null   int64 
 5   repeated_issue   1000 non-null   int64 
 6   after_hours      1000 non-null   int64 
 7   is_breach        1000 non-null   int64 
dtypes: int64(4), object(4)
memory usage: 62.6+ KB


no null values and the is_breach,after_hours,repeated_issue are converted to int datatype,so we can proceed with the next steps 

In [12]:
len(df)

1000

# Convert text categories to numbers using "One-Hot Encoding"
You're performing One-Hot Encoding because machine learning models cannot directly understand text (categorical) values like "Malware", "Network", "Critical", or "High".
They need numerical input.


In [13]:
df_encoded = pd.get_dummies(df,columns=['category', 'priority'],dtype=int)

In [14]:
df_encoded.head(5)

,ticket_id,date,resolution_time,repeated_issue,after_hours,is_breach,category_Access Request,category_Data Export,category_Login Issue,category_Malware Alert,category_Phishing Email,category_Slow Performance,priority_Critical,priority_High,priority_Low,priority_Medium
0,T0001,2023-02-22,4,0,1,1,0,1,0,0,0,0,0,0,1,0
1,T0002,2023-11-05,55,1,1,1,0,1,0,0,0,0,0,0,1,0
2,T0003,2023-01-04,70,0,0,0,0,0,1,0,0,0,0,0,0,1
3,T0004,2023-02-22,44,1,1,0,0,0,0,1,0,0,1,0,0,0
4,T0005,2023-01-23,13,0,0,0,0,0,1,0,0,0,1,0,0,0


automatically creates new column for each category,priority

# STEP 2: DEFINE FEATURES (X) AND TARGET (y)
X = the information we USE to predict
y = what we WANT to predict

#Dropping columns which we don't use in prediction

In [15]:
x = df_encoded.drop(['ticket_id','date','is_breach'],axis = 1)

here in x we dont require ticket_id, date so we are dropping those columns and if we want to predict the is_breach_tickets so we dropped it from x as x is the information we use to predict.
and axis = 1, tells that its a column ; axis=0 is a row

In [16]:
y = df_encoded['is_breach']

In [17]:
x.head(5)

,resolution_time,repeated_issue,after_hours,category_Access Request,category_Data Export,category_Login Issue,category_Malware Alert,category_Phishing Email,category_Slow Performance,priority_Critical,priority_High,priority_Low,priority_Medium
0,4,0,1,0,1,0,0,0,0,0,0,1,0
1,55,1,1,0,1,0,0,0,0,0,0,1,0
2,70,0,0,0,0,1,0,0,0,0,0,0,1
3,44,1,1,0,0,0,1,0,0,1,0,0,0
4,13,0,0,0,0,1,0,0,0,1,0,0,0


In [18]:
x.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 13 columns):
 #   Column                     Non-Null Count  Dtype
---  ------                     --------------  -----
 0   resolution_time            1000 non-null   int64
 1   repeated_issue             1000 non-null   int64
 2   after_hours                1000 non-null   int64
 3   category_Access Request    1000 non-null   int64
 4   category_Data Export       1000 non-null   int64
 5   category_Login Issue       1000 non-null   int64
 6   category_Malware Alert     1000 non-null   int64
 7   category_Phishing Email    1000 non-null   int64
 8   category_Slow Performance  1000 non-null   int64
 9   priority_Critical          1000 non-null   int64
 10  priority_High              1000 non-null   int64
 11  priority_Low               1000 non-null   int64
 12  priority_Medium            1000 non-null   int64
dtypes: int64(13)
memory usage: 101.7 KB


In [19]:
print("Features used for prediction:", x.columns.tolist())

Features used for prediction: ['resolution_time', 'repeated_issue', 'after_hours', 'category_Access Request', 'category_Data Export', 'category_Login Issue', 'category_Malware Alert', 'category_Phishing Email', 'category_Slow Performance', 'priority_Critical', 'priority_High', 'priority_Low', 'priority_Medium']


In [20]:
y.head(5)

0    1
1    1
2    0
3    0
4    0
Name: is_breach, dtype: int64

In [21]:
print(f"\nTotal samples: {len(x)}")


Total samples: 1000


In [22]:
print(f"Breaches: {y.sum()} ({y.mean()*100:.1f}%)")

Breaches: 276 (27.6%)


In [23]:
#Train & Test Split
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.25,random_state = 42)

In [24]:
len(x_train),len(x_test)

(750, 250)

here as we know we have 1000 rows that is splitted into test and train 750 are training data and 250 are testing data

In [25]:
print(x_train.select_dtypes(include='object').columns)

Index([], dtype='object')


In [26]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 8 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   ticket_id        1000 non-null   object
 1   date             1000 non-null   object
 2   category         1000 non-null   object
 3   priority         1000 non-null   object
 4   resolution_time  1000 non-null   int64 
 5   repeated_issue   1000 non-null   int64 
 6   after_hours      1000 non-null   int64 
 7   is_breach        1000 non-null   int64 
dtypes: int64(4), object(4)
memory usage: 62.6+ KB


In [27]:
print(f"\nTraining set: {len(x_train)} rows")
print(f"Testing set: {len(x_test)} rows")


Training set: 750 rows
Testing set: 250 rows


# STEP 4: TRAIN THE MODEL

Random Forest = Many decision trees working together
Think: instead of 1 expert, ask 100 experts and take majority vote

In [28]:
from sklearn.ensemble import RandomForestClassifier

In [29]:
#intiate the classifier
model_rf = RandomForestClassifier()
model_rf= RandomForestClassifier(n_estimators=100,random_state=42,class_weight='balanced')  # 100 decision trees
# passing the training data to the model
model_rf.fit(x_train, y_train);


In [30]:
print("\n✅ Model trained!")


✅ Model trained!


# STEP 5: EVALUATE THE MODEL

In [31]:
y_pred = model_rf.predict(x_test) 

In [32]:
y_pred

array([1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 1, 0, 0, 0, 0,
       0, 0, 0, 1, 1, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 0, 0, 1, 1, 0, 1, 0, 0, 1, 0, 0, 0,
       0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 1,
       0, 0, 0, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 1, 1, 0,
       0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 1, 0, 0, 1, 1, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0,
       1, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0,
       0, 0, 1, 0, 0, 1, 1, 0])

In [33]:
print("\n📊 MODEL PERFORMANCE REPORT:")


📊 MODEL PERFORMANCE REPORT:


In [36]:
from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred, target_names=['Normal Ticket', 'Breach']))

               precision    recall  f1-score   support

Normal Ticket       0.73      0.80      0.76       177
       Breach       0.36      0.27      0.31        73

     accuracy                           0.64       250
    macro avg       0.54      0.54      0.54       250
 weighted avg       0.62      0.64      0.63       250



# Precision = of predicted breaches, how many were ACTUALLY breaches?
# Recall    = of actual breaches, how many did we CATCH?
# F1 Score  = balance between precision and recall

In [37]:
print(y.value_counts())

is_breach
0    724
1    276
Name: count, dtype: int64


In [38]:
from sklearn.model_selection import train_test_split
x_xgb_train,x_xgb_test,y_xgb_train,y_xgb_test = train_test_split(x,y,test_size=0.25,random_state = 42)

In [39]:
import xgboost as xgb
from xgboost import XGBClassifier

In [40]:
model_xgb = XGBClassifier(n_estimators=100,learning_rate=0.1,max_depth=4,random_state=42)
model_xgb.fit(x_xgb_train, y_xgb_train);

In [41]:
print("\n✅ Model 2 trained!")


✅ Model 2 trained!


In [42]:
# EVALUATE THE MODEL
y_pred_xgb = model_xgb.predict(x_xgb_test) 

In [43]:
y_pred_xgb

array([1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0,
       0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 1, 0, 0,
       0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 1, 0, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0,
       1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0,
       0, 1, 0, 0, 1, 0, 1, 0])

In [44]:
print(classification_report(y_xgb_test, y_pred_xgb, target_names=['Normal Ticket', 'Breach']))

               precision    recall  f1-score   support

Normal Ticket       0.72      0.88      0.79       177
       Breach       0.39      0.19      0.26        73

     accuracy                           0.68       250
    macro avg       0.56      0.53      0.52       250
 weighted avg       0.63      0.68      0.64       250



"To improve prediction performance, I implemented an XGBoost classifier and compared it with Random Forest models. The XGBoost model achieved the highest overall accuracy of 68%, outperforming the other approaches. It was particularly effective at identifying normal tickets, achieving an 88% recall for the Normal Ticket class."

"However, when evaluating the Breach class, which is the primary business concern, the model achieved a recall of only 19%. This means that while the model was accurate overall, it still missed a large number of actual SLA breaches."

"This analysis highlighted an important business insight: optimizing only for accuracy can be misleading. Although XGBoost produced the highest accuracy, it was not the best model for identifying breach tickets. In comparison, Random Forest achieved a higher breach recall of 29%".

"Based on these findings, I concluded that model selection should be driven by business objectives rather than accuracy alone. For SLA breach prediction, detecting high-risk tickets is more valuable than maximizing overall accuracy."

# "Although XGBoost achieved the highest accuracy of 68%, the business objective was to identify potential SLA breaches. Random Forest achieved a higher recall for breach tickets (29% versus 19%), meaning it detected more high-risk cases. Since preventing SLA violations is more important than maximizing overall accuracy, I selected Random Forest as the preferred model."

Top Breach Predictors (Feature Importance)

resolution_time          — longer resolution = higher risk
category_Malware Alert   — highest single-category risk
category_Phishing Email  — second most dangerous
priority_Critical        — escalates breach probability significantly
after_hours              — night-time tickets carry extra risk